# 🍊 Train YOLO11 — Orange Detector (CVAT Corrected)

Upload your corrected CVAT annotations + frames, then train YOLO11.

**Runtime:** `Runtime > Change runtime type > GPU (T4)`

## 1. Install

In [ ]:
!pip install ultralytics opencv-python pyyaml -q

In [ ]:
import os
import cv2
import glob
import shutil
import random
import zipfile
import subprocess
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from google.colab import files

DATASET_DIR = "dataset"

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload CVAT annotations (zip exported in YOLO 1.1)

In [ ]:
print("Upload your ZIP exported from CVAT:")
uploaded_cvat = files.upload()
cvat_zip = list(uploaded_cvat.keys())[0]

os.makedirs(DATASET_DIR, exist_ok=True)

with zipfile.ZipFile(cvat_zip, "r") as zf:
    zf.extractall(DATASET_DIR)

# Count labels
all_labels = glob.glob(f"{DATASET_DIR}/**/*.txt", recursive=True)
all_labels = [l for l in all_labels if os.path.basename(l) not in ["obj.data", "train.txt", "valid.txt"]]
print(f"\n✅ {len(all_labels)} annotation files extracted")

## 3. Upload the frames (images ZIP file)

In [ ]:
print("Upload your frames ZIP file:")
uploaded_frames = files.upload()
frames_zip = list(uploaded_frames.keys())[0]

# Find the folder that already contains the labels
img_dest = None
for root, dirs, fls in os.walk(DATASET_DIR):
    if any(f.endswith(".txt") and f not in ["obj.data", "train.txt", "valid.txt"] for f in fls):
        img_dest = root
        break

if img_dest is None:
    img_dest = f"{DATASET_DIR}/obj_train_data"
    os.makedirs(img_dest, exist_ok=True)

with zipfile.ZipFile(frames_zip, "r") as zf:
    for f in zf.namelist():
        if f.endswith((".jpg", ".png", ".jpeg")):
            fname = os.path.basename(f)
            if fname:  # skip directories
                with zf.open(f) as src, open(os.path.join(img_dest, fname), "wb") as dst:
                    dst.write(src.read())

imgs = glob.glob(f"{img_dest}/*.jpg") + glob.glob(f"{img_dest}/*.png") + glob.glob(f"{img_dest}/*.jpeg")
print(f"\n✅ {len(imgs)} frames extraites dans {img_dest}")

## 4. Split 80/20

In [ ]:
all_imgs = sorted(glob.glob(f"{img_dest}/*.jpg") + glob.glob(f"{img_dest}/*.png") + glob.glob(f"{img_dest}/*.jpeg"))
print(f"Total: {len(all_imgs)} images")

# Check how many have a label
with_labels = 0
for img_path in all_imgs:
    label_name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
    if os.path.exists(os.path.join(img_dest, label_name)):
        with_labels += 1
print(f"Avec annotations: {with_labels}/{len(all_imgs)}")

for d in ["train/images", "train/labels", "valid/images", "valid/labels"]:
    os.makedirs(f"{DATASET_DIR}/{d}", exist_ok=True)

random.seed(42)
random.shuffle(all_imgs)
split_idx = int(len(all_imgs) * 0.8)

for i, img_path in enumerate(all_imgs):
    split = "train" if i < split_idx else "valid"
    fname = os.path.basename(img_path)
    label_name = os.path.splitext(fname)[0] + ".txt"
    label_path = os.path.join(img_dest, label_name)

    shutil.copy(img_path, f"{DATASET_DIR}/{split}/images/{fname}")
    if os.path.exists(label_path):
        shutil.copy(label_path, f"{DATASET_DIR}/{split}/labels/{label_name}")

train_count = len(glob.glob(f"{DATASET_DIR}/train/images/*"))
valid_count = len(glob.glob(f"{DATASET_DIR}/valid/images/*"))
print(f"\n✅ Split done!")
print(f"  Train: {train_count} images")
print(f"  Valid: {valid_count} images")

## 5. Preview annotations

In [ ]:
sample_imgs = sorted(glob.glob(f"{DATASET_DIR}/train/images/*"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 16))
for ax, img_path in zip(axes.flat, sample_imgs):
    img = cv2.imread(img_path)
    h_img, w_img = img.shape[:2]
    fname = os.path.basename(img_path)
    label_path = f"{DATASET_DIR}/train/labels/{os.path.splitext(fname)[0]}.txt"
    n_labels = 0
    if os.path.exists(label_path):
        with open(label_path) as f:
            lines = f.readlines()
            n_labels = len(lines)
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = int((xc - bw / 2) * w_img)
                    y1 = int((yc - bh / 2) * h_img)
                    x2 = int((xc + bw / 2) * w_img)
                    y2 = int((yc + bh / 2) * h_img)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{fname} ({n_labels} oranges)", fontsize=9)
    ax.axis("off")
plt.suptitle("CVAT Corrected Annotations", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Train YOLO26n 🚀

In [ ]:
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

data_config = {
    "train": os.path.abspath(f"{DATASET_DIR}/train/images"),
    "val": os.path.abspath(f"{DATASET_DIR}/valid/images"),
    "nc": 1,
    "names": ["orange"],
}

with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f)

print("✅ data.yaml created")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=data_yaml_path,
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    save=True,
    project="clementine_training",
    name="yolo11_clementine",
    exist_ok=True,
)

print("\n✅ Training complete!")

## 7. Evaluate

In [ ]:
result = subprocess.run(["find", ".", "-path", "*/yolo26n_clementine/weights/best.pt"],
                        capture_output=True, text=True)
best_model_path = result.stdout.strip().split("\n")[0]
print(f"Model: {best_model_path}")

model = YOLO(best_model_path)
metrics = model.val()

print(f"\nmAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Training curves
results_dir = os.path.dirname(os.path.dirname(best_model_path))
results_img = os.path.join(results_dir, "results.png")
if os.path.exists(results_img):
    display(Image(filename=results_img, width=800))

In [ ]:
# Confusion matrix
cm_img = os.path.join(results_dir, "confusion_matrix.png")
if os.path.exists(cm_img):
    display(Image(filename=cm_img, width=600))

In [ ]:
# Test predictions on validation images
model = YOLO(best_model_path)
test_imgs = sorted(glob.glob(f"{DATASET_DIR}/valid/images/*"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 16))
for ax, img_path in zip(axes.flat, test_imgs):
    preds = model(img_path, verbose=False)
    annotated = preds[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{len(preds[0].boxes)} detections", fontsize=10)
    ax.axis("off")
plt.suptitle("YOLO11 Predictions", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Download best.pt ⬇️

Then run locally:
```bash
python orange_counter.py --source orange_video.mp4 --model best.pt --interactive --save
```

In [ ]:
print(f"Model: {best_model_path}")
print(f"Size: {os.path.getsize(best_model_path) / 1e6:.1f} MB")
files.download(best_model_path)
print("\n✅ Download started!")

## 9. (Optional) Save to Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
drive_dest = "/content/drive/MyDrive/clementine_model/"
os.makedirs(drive_dest, exist_ok=True)
shutil.copy(best_model_path, drive_dest)
print(f"✅ Saved to: {drive_dest}")